# 📖 TasmiqAI — Quran Expert System (Gradio)

Run **Cell 1** once → **Restart kernel** → Run **Cell 2**.

## Cell 1 — Install

In [ ]:
import sys
!{sys.executable} -m pip install gradio soundfile librosa transformers torch --quiet
print('✅ Ready. Run Cell 2.')

## Cell 2 — Launch UI
Loads dataset + model, opens Gradio in browser.

In [ ]:
import json, logging, traceback
import numpy as np
from pathlib import Path
import soundfile as sf
import librosa
import torch
import gradio as gr
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

logging.basicConfig(level=logging.WARNING)

# ── Dataset ──────────────────────────────────────────────────────────────
DATASET_ROOT = Path(r'C:\Users\nabil\.gemini\antigravity\scratch\quranjson\source')
AUDIO_ROOT   = DATASET_ROOT / 'audio'

print('📂 Loading dataset...')
quran = {}
for _sf in sorted((DATASET_ROOT/'surah').glob('surah_*.json'),
                   key=lambda p: int(p.stem.split('_')[1])):
    with open(_sf, 'r', encoding='utf-8') as _f:
        _d = json.load(_f)
    quran[int(_d['index'])] = _d

with open(DATASET_ROOT/'juz.json', 'r', encoding='utf-8') as _f:
    juz_data = json.load(_f)

total_v = sum(s['count'] for s in quran.values())
print(f'✅ {len(quran)} surahs · {total_v} verses · {len(juz_data)} juz')

def get_ayah(s, a):  return quran.get(s,{}).get('verse',{}).get(f'verse_{a}','[not found]')
def get_name(s):     return quran.get(s,{}).get('name','Unknown')
def get_count(s):    return quran.get(s,{}).get('count',0)
def audio_p(s, a):  return AUDIO_ROOT / f'{s:03d}' / f'{a:03d}.mp3'

# ── Model ─────────────────────────────────────────────────────────────────
print('⏳ Loading Wav2Vec2 model...')
_proc  = Wav2Vec2Processor.from_pretrained('TBOGamer22/wav2vec2-quran-phonetics')
_model = Wav2Vec2ForCTC.from_pretrained('TBOGamer22/wav2vec2-quran-phonetics')
_model.eval()
_dev = 'cuda' if torch.cuda.is_available() else 'cpu'
_model.to(_dev)
print(f'✅ Model on: {_dev}')

# ── Audio loader — tries soundfile then librosa ───────────────────────────
def load_audio(path_str):
    try:
        arr, sr = sf.read(path_str, dtype='float32')
        if arr.ndim > 1:
            arr = arr.mean(axis=1)
        if sr != 16000:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        return arr
    except Exception:
        # fallback to librosa
        arr, _ = librosa.load(path_str, sr=16000, mono=True)
        return arr

# ── Assessment function ───────────────────────────────────────────────────
def assess(surah_label, ayah_num):
    try:
        s = int(str(surah_label).split('.')[0].strip())
        a = int(float(ayah_num))

        arabic = get_ayah(s, a)
        name   = get_name(s)
        ap     = audio_p(s, a)
        label  = f'Surah {s} — {name}  |  Ayah {a}'

        if not ap.exists():
            return label, arabic, '⚠️ No audio in dataset for this ayah.', None

        # Load audio
        arr = load_audio(str(ap))

        # Phonetic inference
        inp = _proc(arr, sampling_rate=16000, return_tensors='pt', padding=True)
        inp = {k: v.to(_dev) for k, v in inp.items()}
        with torch.inference_mode():
            logits = _model(**inp).logits
        ph = _proc.batch_decode(torch.argmax(logits, dim=-1),
                                skip_special_tokens=True)[0]

        ref_path = None
        if ap.exists():
            import shutil
            cache_dir = Path('./ref_cache')
            cache_dir.mkdir(exist_ok=True)
            local_path = cache_dir / f'{s:03d}_{a:03d}.mp3'
            if not local_path.exists():
                shutil.copy(str(ap), str(local_path))
            ref_path = str(local_path)

        return label, arabic, ph or '(no output)', ref_path

    except Exception as e:
        err = traceback.format_exc()
        print('ASSESS ERROR:\n', err)
        return f'ERROR: {e}', err, err, None

# ── Surah list ─────────────────────────────────────────────────────────────
surah_choices = [f'{i:03d}. {get_name(i)} ({get_count(i)} ayahs)'
                 for i in range(1, 115)]

# ── Gradio UI ─────────────────────────────────────────────────────────────
with gr.Blocks(title='TasmiqAI', theme=gr.themes.Soft()) as demo:

    gr.HTML(
        "<div style='background:linear-gradient(135deg,#1a472a,#2d6a4f);"
        "color:white;padding:18px 24px;border-radius:12px;margin-bottom:10px'>"
        "<h1 style='margin:0;font-size:22px'>📖 TasmiqAI — Quran Expert System</h1>"
        "<p style='margin:4px 0 0;opacity:.85;font-size:13px'>"
        "Wav2Vec2 Phonetics · 30 Juz · 114 Surahs · 6,236 Verses</p></div>"
    )

    with gr.Row():
        dd_surah = gr.Dropdown(
            choices=surah_choices, value=surah_choices[0],
            label='📂 Select Surah', scale=3)
        nb_ayah = gr.Number(
            value=1, minimum=1, maximum=286,
            label='📜 Ayah Number', precision=0, scale=1)

    btn = gr.Button('▶  Run Assessment', variant='primary', size='lg')

    with gr.Row():
        with gr.Column(scale=3):
            out_label  = gr.Textbox(label='📖 Target', interactive=False)
            out_arabic = gr.Textbox(
                label='📝 Arabic Text', interactive=False, lines=3)
        with gr.Column(scale=2):
            out_audio = gr.Audio(
                label='🔊 Dataset Audio', type='filepath')

    out_ph = gr.Textbox(
        label='🎙️ Phonetics Detected by Wav2Vec2',
        interactive=False, lines=2)

    gr.Markdown(
        "> ℹ️ Model outputs Arabic phoneme tokens. "
        "A transliteration layer is needed for text alignment."
    )

    btn.click(
        fn=assess,
        inputs=[dd_surah, nb_ayah],
        outputs=[out_label, out_arabic, out_ph, out_audio])

print('\n🚀 Launching — browser will open at http://127.0.0.1:7860')
demo.launch(share=False, inbrowser=True)